<a href="https://colab.research.google.com/github/louistrue/learn-ifc-bfh25-D/blob/main/MVP-Massivbau%20Auswertung%20W%C3%A4nde%2023.12.2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install ifcopenshell pandas openpyxl


In [6]:
import requests
import ifcopenshell
import pandas as pd
from ifcopenshell.util.element import get_psets
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font
import datetime

# -----------------------------
# 1. IFC von GitHub laden
# -----------------------------
url = "https://raw.githubusercontent.com/louistrue/learn-ifc-bfh25-D/main/Modelle/BFH-25/2379_230926_V1033_BA_41_SCA_300B.ifc"
local_ifc = "modell.ifc"

r = requests.get(url)
r.raise_for_status()
with open(local_ifc, "wb") as f:
    f.write(r.content)

# -----------------------------
# 2. IFC öffnen und Wände auslesen
# -----------------------------
model = ifcopenshell.open(local_ifc)
walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")

rows = []
for wall in walls:
    psets = get_psets(wall)
    wg_sub = psets.get("wg_SUB", {})
    betontyp = wg_sub.get("Betontyp")
    qto = psets.get("Qto_WallBaseQuantities", {})
    hoehe = qto.get("Height")
    volumen = qto.get("NetVolume")
    flaeche = qto.get("NetSideArea")
    wanddicke = qto.get("Width") # Extract wall thickness

    rows.append({
        "GlobalId": wall.GlobalId,
        "Betontyp": betontyp,
        "Wandhoehe_m": hoehe,
        "Wanddicke_m": wanddicke,
        "Volumen_m3": volumen,
        "Flaeche_m2": flaeche
    })

df = pd.DataFrame(rows).dropna(subset=["Wandhoehe_m", "Volumen_m3"])

# -----------------------------
# 3. Betontyp → NPK
# -----------------------------
BETONTYP_TO_NPK = {
    "01": "NPK C",
    "02": "NPK B",
    "03": "NPK A"
}
df["NPK"] = df["Betontyp"].map(BETONTYP_TO_NPK)

# -----------------------------
# 4. Wandhöhe-Kategorien
# -----------------------------
def hoehe_kategorie(h):
    if h <= 1.5:
        return "bis 1.5m"
    elif 1.51 <= h <= 1.99:
        return "1.51-1.99m"
    elif 2.0 <= h <= 2.99:
        return "2.0-2.99m"
    elif 3.00 <= h <= 4.00:
        return "3.00-4.00m"
    else:
        return "größer 4.0m"

df["Hoehe_Kategorie"] = df["Wandhoehe_m"].apply(hoehe_kategorie)

# -----------------------------
# 5. Gruppierte Auswertung
# -----------------------------
df_grouped = df.groupby(["NPK", "Hoehe_Kategorie"]).agg(
    Anzahl_Waende=("GlobalId", "count"),
    Gesamtvolumen_m3=("Volumen_m3", "sum"),
    Gesamtflaeche_m2=("Flaeche_m2", "sum")
).reset_index()

# -----------------------------
# 6. Excel-Ausgabe
# -----------------------------
# Get current date and time for filename
current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"NPK_Waende_Auswertung_{current_time}.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Einzelwände
    df.to_excel(writer, sheet_name="Einzelwaende", index=False)
    worksheet_einzelwaende = writer.sheets["Einzelwaende"]
    max_row_einzelwaende = len(df) + 1
    max_col_einzelwaende = get_column_letter(len(df.columns))
    worksheet_einzelwaende.auto_filter.ref = f"A1:{max_col_einzelwaende}{max_row_einzelwaende}"

    # Add total row for Einzelwaende
    total_row_einzelwaende = max_row_einzelwaende + 1
    worksheet_einzelwaende[f'A{total_row_einzelwaende}'] = 'Total'
    worksheet_einzelwaende[f'A{total_row_einzelwaende}'].font = Font(bold=True)

    # Columns to sum for Einzelwaende: Volumen_m3 (E), Flaeche_m2 (F)
    volumen_col_einzelwaende = get_column_letter(df.columns.get_loc('Volumen_m3') + 1)
    flaeche_col_einzelwaende = get_column_letter(df.columns.get_loc('Flaeche_m2') + 1)

    worksheet_einzelwaende[f'{volumen_col_einzelwaende}{total_row_einzelwaende}'] = f"=SUBTOTAL(109, {volumen_col_einzelwaende}2:{volumen_col_einzelwaende}{max_row_einzelwaende})"
    worksheet_einzelwaende[f'{volumen_col_einzelwaende}{total_row_einzelwaende}'].font = Font(bold=True)
    worksheet_einzelwaende[f'{flaeche_col_einzelwaende}{total_row_einzelwaende}'] = f"=SUBTOTAL(109, {flaeche_col_einzelwaende}2:{flaeche_col_einzelwaende}{max_row_einzelwaende})"
    worksheet_einzelwaende[f'{flaeche_col_einzelwaende}{total_row_einzelwaende}'].font = Font(bold=True)

    # Gruppierte Übersicht
    df_grouped.to_excel(writer, sheet_name="Auswertung_NPK_Hoehe", index=False)
    worksheet_grouped = writer.sheets["Auswertung_NPK_Hoehe"]
    max_row_grouped = len(df_grouped) + 1
    max_col_grouped = get_column_letter(len(df_grouped.columns))
    worksheet_grouped.auto_filter.ref = f"A1:{max_col_grouped}{max_row_grouped}"

    # Add total row for Auswertung_NPK_Hoehe
    total_row_grouped = max_row_grouped + 1
    worksheet_grouped[f'A{total_row_grouped}'] = 'Total'
    worksheet_grouped[f'A{total_row_grouped}'].font = Font(bold=True)

    # Columns to sum for Auswertung_NPK_Hoehe: Anzahl_Waende (C), Gesamtvolumen_m3 (D), Gesamtflaeche_m2 (E)
    # These indices should remain the same for df_grouped
    anzahl_col_grouped = get_column_letter(df_grouped.columns.get_loc('Anzahl_Waende') + 1)
    volumen_col_grouped = get_column_letter(df_grouped.columns.get_loc('Gesamtvolumen_m3') + 1)
    flaeche_col_grouped = get_column_letter(df_grouped.columns.get_loc('Gesamtflaeche_m2') + 1)

    worksheet_grouped[f'{anzahl_col_grouped}{total_row_grouped}'] = f"=SUBTOTAL(109, {anzahl_col_grouped}2:{anzahl_col_grouped}{max_row_grouped})"
    worksheet_grouped[f'{anzahl_col_grouped}{total_row_grouped}'].font = Font(bold=True)
    worksheet_grouped[f'{volumen_col_grouped}{total_row_grouped}'] = f"=SUBTOTAL(109, {volumen_col_grouped}2:{volumen_col_grouped}{max_row_grouped})"
    worksheet_grouped[f'{volumen_col_grouped}{total_row_grouped}'].font = Font(bold=True)
    worksheet_grouped[f'{flaeche_col_grouped}{total_row_grouped}'] = f"=SUBTOTAL(109, {flaeche_col_grouped}2:{flaeche_col_grouped}{max_row_grouped})"
    worksheet_grouped[f'{flaeche_col_grouped}{total_row_grouped}'].font = Font(bold=True)


print(f"Neue Excel-Datei erstellt: {output_path}")

Neue Excel-Datei erstellt: NPK_Waende_Auswertung_20260106_083715.xlsx
